# `plotting` — Diagnostic Report

One 2×2 diagnostic figure per factor, saved to `config.OUTPUT_DIR` (the project-root `/output`). The four panels are exactly the four diagnostics a PM asks for:

| Panel | What it answers |
|---|---|
| **Cumulative IC** (raw vs neutralized) | Is the edge persistent, or a few lucky days? A steady up-slope = a real, stable signal. |
| **IC decay curve** | How fast does the edge die? A curve that drops off quickly = you must trade often to capture it. |
| **Long-short equity, gross vs net** | Does the edge survive costs? The gap between the two lines *is* the cost drag. |
| **Per-quintile annualized return** | Is the relationship monotonic? A clean staircase Q1→Q5 = a trustworthy sort. |

*(Function definition only — `main.ipynb` calls it after the pipeline runs.)*

In [ ]:
"""Diagnostic plotting: a 2x2 research report per factor."""
import os
import matplotlib.pyplot as plt
from typing import Dict


def generate_report_plots(metrics_raw: Dict, metrics_neut: Dict, factor_name: str,
                          output_dir: str = "../output") -> str:
    """Render and save the 2x2 diagnostic report for one factor. Returns the saved path."""
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle(f'Research Diagnostic Report - Alpha Factor: [{factor_name}]',
                 fontsize=16, fontweight='bold')

    # Panel 1 (top-left): cumulative IC, raw vs neutralized. A rising line = a persistent edge.
    axes[0, 0].plot(metrics_raw['cum_ic_curve'], label='Raw Factor', color='crimson', alpha=0.7)
    axes[0, 0].plot(metrics_neut['cum_ic_curve'], label='Neutralized Factor', color='navy', lw=2)
    axes[0, 0].set_title('Cumulative Information Coefficient')
    axes[0, 0].grid(True, ls='--', alpha=0.5); axes[0, 0].legend()

    # Panel 2 (top-right): IC decay vs forward horizon. Pull the stored decay dict into lists.
    horizons = [1, 2, 3, 5, 10]
    raw_decay = [metrics_raw['ic_decay'][f'Horizon_{h}'] for h in horizons]
    neut_decay = [metrics_neut['ic_decay'][f'Horizon_{h}'] for h in horizons]
    axes[0, 1].plot(horizons, raw_decay, marker='o', ls='--', color='crimson', label='Raw Alpha')
    axes[0, 1].plot(horizons, neut_decay, marker='s', ls='-', color='navy', label='Neutralized Alpha')
    axes[0, 1].axhline(0, color='grey', lw=0.8)
    axes[0, 1].set_title('IC Decay Curve (Alpha Half-Life)')
    axes[0, 1].set_xlabel('Forward Horizon (trading days)')
    axes[0, 1].set_ylabel('Mean Spearman Rank IC')
    axes[0, 1].grid(True, ls='--', alpha=0.5); axes[0, 1].legend()

    # Panel 3 (bottom-left): the money shot. Gross vs net long-short equity for the neutralized
    # book -- the widening gap between the two lines is the transaction-cost drag.
    axes[1, 0].plot(metrics_neut['equity_curve_gross'], color='teal', ls='--',
                    label='Long-Short (Gross)')
    axes[1, 0].plot(metrics_neut['equity_curve_net'], color='darkgreen', lw=2,
                    label='Long-Short (Net of Cost)')
    axes[1, 0].axhline(0, color='grey', lw=0.8)
    axes[1, 0].set_title('Long-Short Equity Curve: Gross vs Net of Cost')
    axes[1, 0].grid(True, ls='--', alpha=0.5); axes[1, 0].legend()

    # Panel 4 (bottom-right): annualized return of each quintile. A clean staircase up to Q5
    # means the sort is monotonic and trustworthy.
    quantiles = list(metrics_neut['quantile_returns'].index)
    axes[1, 1].bar(quantiles, metrics_neut['quantile_returns'].values,
                   color='cornflowerblue', edgecolor='black')
    axes[1, 1].set_title('Annualized Return by Quantile')
    axes[1, 1].set_xlabel('Quantile (1 = short tail, 5 = long head)')
    axes[1, 1].set_ylabel('Annualized Return')
    axes[1, 1].grid(True, ls='--', alpha=0.3)

    plt.tight_layout()
    out_path = os.path.join(output_dir, f'alpha_diagnostic_{factor_name.lower()}.png')
    plt.savefig(out_path, dpi=300)
    plt.show()
    plt.close(fig)
    print(f'[saved] {out_path}')
    return out_path


print("plotting helper ready: generate_report_plots()")